<a href="https://colab.research.google.com/github/MurphyKlein/CS4782_final_project/blob/main/notebook/patchtst_supervised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PatchTST — Supervised Replication (Table 4, Weather dataset)

Replicates the **Supervised PatchTST/42** column of Table 4 from:
> *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers* (ICLR 2023)

**Config (from paper):**
- Look-back window `L = 336`, patch length `P = 16`, stride `S = 8` → **42 patches**
- Channel-independence: each of the 21 Weather channels processed independently
- BatchNorm (not LayerNorm), instance normalisation, learnable positional encoding
- Prediction horizons `T ∈ {96, 192, 336, 720}`, metric: MSE + MAE

**Paper targets (Weather):**

| T   | MSE   | MAE   |
|-----|-------|-------|
| 96  | 0.152 | 0.199 |
| 192 | 0.197 | 0.243 |
| 336 | 0.249 | 0.283 |
| 720 | 0.320 | 0.335 |

## Cell 1 — Imports & reproducibility

In [ ]:
import os, math, json, time, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 2021
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : Tesla T4


## Cell 2 — Configuration

In [ ]:
from google.colab import drive
import os

# Standard simple mount
drive.mount('/content/drive')

# Let's check if the project folder exists
project_path = '/content/drive/MyDrive/DL_Final_Project'
if os.path.exists(project_path):
    print(f"Success! Connected to: {project_path}")
    print("Folders found:", os.listdir(project_path))
else:
    print(f"Connected to Drive, but could not find {project_path}")

Mounted at /content/drive
Success! Connected to: /content/drive/MyDrive/DL_Final_Project
Folders found: ['supervised.ipynb', 'Lin_Prob.ipynb', 'Fine_Tuning.ipynb', 'fine.ipynb', 'main.ipynb', 'Playground.ipynb', 'Visual-Graphs.ipynb', 'weather_data', 'electricity_data']


In [ ]:
import os

weather_data_path = os.path.join(project_path, 'weather_data')

print(f"Checking for weather_data directory at: {weather_data_path}")
if os.path.exists(weather_data_path):
    print(f"'weather_data' directory found. Contents: {os.listdir(weather_data_path)}")
    print("If this list is empty or doesn't contain CSV files, please ensure your weather data files are uploaded here.")
else:
    print(f"'weather_data' directory NOT found at {weather_data_path}.")
    print("Please create a folder named 'weather_data' inside your 'DL_Final_Project' directory in Google Drive and upload your CSV files there.")
    print("Alternatively, if your data is directly in 'DL_Final_Project', you might need to adjust `WEATHER_DIR` in Cell 2.")


Checking for weather_data directory at: /content/drive/MyDrive/DL_Final_Project/weather_data
'weather_data' directory found. Contents: ['mpi_roof_2025.csv']
If this list is empty or doesn't contain CSV files, please ensure your weather data files are uploaded here.


In [ ]:
import math
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/DL_Final_Project')
WEATHER_DIR = BASE_DIR / 'weather_data'
CKPT_DIR    = BASE_DIR / 'checkpoints'     # model checkpoints saved here
RESULTS_DIR = BASE_DIR / 'results'         # JSON results saved here
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── PatchTST/42 hyperparameters (Appendix A.1.4) ─────────────────────
L       = 336    # look-back window
P       = 16     # patch length
S       = 8      # stride

D_MODEL  = 128
N_HEADS  = 16
N_LAYERS = 3
D_FF     = 256
DROPOUT  = 0.2

# ── Training hyperparameters ──────────────────────────────────────────
BATCH_SIZE  = 128
LR          = 1e-4
MAX_EPOCHS  = 100
PATIENCE    = 10
LR_PATIENCE = 5
LR_MIN      = 1e-6

# ── Prediction horizons to sweep ─────────────────────────────────────
HORIZONS = [96, 192, 336, 720]

# ── Data split ratios ─────────────────────────────────────────────────
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1

print(f'Config set to: {WEATHER_DIR}')
N_patches = math.floor((L - P) / S) + 2
print(f'N patches = {N_patches}')

Config set to: /content/drive/MyDrive/DL_Final_Project/weather_data
N patches = 42


## Cell 3 — Load & merge Weather data

In [ ]:
import glob
import pandas as pd
import numpy as np
import os

# Debugging step: List files to find where the CSVs are
print(f"Searching for CSVs in: {WEATHER_DIR}")
if WEATHER_DIR.exists():
    print("Contents of directory:", os.listdir(WEATHER_DIR))
else:
    print("Directory does not exist!")

def load_weather(directory: Path) -> np.ndarray:
    all_files = sorted(glob.glob(str(directory / '*.csv')))
    if not all_files:
        # Try one level deeper just in case of nested folders
        all_files = sorted(glob.glob(str(directory / '*' / '*.csv')))

    if not all_files:
        raise FileNotFoundError(f'No CSV files found in {directory}. Please check your Drive path.')

    frames = []
    for fp in all_files:
        df = pd.read_csv(fp, encoding='unicode_escape', low_memory=False)
        print(f'  Loaded {Path(fp).name:40s} shape={df.shape}')
        date_col = next((c for c in df.columns if 'date' in c.lower()), None)
        if date_col:
            df[date_col] = pd.to_datetime(df[date_col], format='mixed', dayfirst=True)
            df = df.set_index(date_col)
        frames.append(df.select_dtypes(include=[np.number]))

    combined = pd.concat(frames, axis=0).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    print(f'\nFinal combined shape: {combined.shape}')
    return combined.values.astype(np.float32)

try:
    weather_data = load_weather(WEATHER_DIR)
    M = weather_data.shape[1]
    print(f'Number of channels (M) = {M}')
except Exception as e:
    print(f'Error loading data: {e}')

Searching for CSVs in: /content/drive/MyDrive/DL_Final_Project/weather_data
Contents of directory: ['mpi_roof_2025.csv']
  Loaded mpi_roof_2025.csv                        shape=(52560, 22)

Final combined shape: (52560, 21)
Number of channels (M) = 21


## Cell 4 — Dataset & DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    """
    Sliding-window dataset for multivariate time series.

    Returns
    -------
    x : FloatTensor (M, L)  — look-back window, channel-first
    y : FloatTensor (M, T)  — forecast target, channel-first
    """

    def __init__(self, data: np.ndarray, L: int, T: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.L = L
        self.T = T

    def __len__(self) -> int:
        return len(self.data) - self.L - self.T + 1

    def __getitem__(self, idx: int):
        x = self.data[idx          : idx + self.L]           # (L, M)
        y = self.data[idx + self.L : idx + self.L + self.T]  # (T, M)
        return x.T, y.T   # → (M, L), (M, T)  channel-first


def build_loaders(data: np.ndarray, L: int, T: int) -> tuple:
    """
    Splits data → train / val / test, fits a StandardScaler on train,
    and returns (train_loader, val_loader, test_loader, scaler).
    """
    n       = len(data)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    train_raw = data[:n_train]
    val_raw   = data[n_train : n_train + n_val]
    test_raw  = data[n_train + n_val :]

    # Fit scaler ONLY on training data to avoid leakage
    scaler    = StandardScaler().fit(train_raw)
    train_sc  = scaler.transform(train_raw)
    val_sc    = scaler.transform(val_raw)
    test_sc   = scaler.transform(test_raw)

    kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    train_dl = DataLoader(TimeSeriesDataset(train_sc, L, T), shuffle=True,  **kw)
    val_dl   = DataLoader(TimeSeriesDataset(val_sc,   L, T), shuffle=False, **kw)
    test_dl  = DataLoader(TimeSeriesDataset(test_sc,  L, T), shuffle=False, **kw)

    return train_dl, val_dl, test_dl, scaler


# Quick sanity-check with T=96
if 'weather_data' in locals() or 'weather_data' in globals():
    _tr, _va, _te, _sc = build_loaders(weather_data, L, T=96)
    x0, y0 = next(iter(_tr))
    print(f'x shape : {tuple(x0.shape)}  (batch, channels, L)')
    print(f'y shape : {tuple(y0.shape)}  (batch, channels, T)')
    print(f'Train batches={len(_tr)}  Val={len(_va)}  Test={len(_te)}')
else:
    print("weather_data is not defined. Please fix the data loading step.")

x shape : (128, 21, 336)  (batch, channels, L)
y shape : (128, 21, 96)  (batch, channels, T)
Train batches=285  Val=38  Test=79


## Cell 5 — Patching utility

In [ ]:
def make_patches(x: torch.Tensor, P: int, S: int) -> torch.Tensor:
    """
    Segment a batch of channel-first time series into overlapping patches.

    Parameters
    ----------
    x : (B, M, L)   — batch × channels × look-back
    P : patch length
    S : stride  (P > S → overlapping patches)

    Returns
    -------
    patches : (B*M, N, P)
        N = floor((L - P) / S) + 2  (paper eq., last value padded by S steps)

    The Transformer then treats each patch as a single input token.
    """
    B, M, L_seq = x.shape

    # Pad the tail with S repetitions of the final value so the last patch
    # always uses real data (not zeros)
    pad = x[:, :, -1:].expand(-1, -1, S)       # (B, M, S)
    x_pad = torch.cat([x, pad], dim=-1)         # (B, M, L+S)

    # unfold → (B, M, N, P)
    patches = x_pad.unfold(dimension=2, size=P, step=S)
    N = patches.shape[2]

    # Merge batch and channel dims so the Transformer sees (B*M) independent series
    return patches.reshape(B * M, N, P)


# Verify expected N
_dummy = torch.zeros(2, M, L)
_p     = make_patches(_dummy, P, S)
print(f'make_patches test: input (2, {M}, {L}) → patches {tuple(_p.shape)}')
print(f'Expected N = {math.floor((L-P)/S)+2},  Got N = {_p.shape[1]}')

make_patches test: input (2, 21, 336) → patches (42, 42, 16)
Expected N = 42,  Got N = 42


## Cell 6 — Model definition

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Transformer encoder layer
# Uses BatchNorm instead of LayerNorm — paper footnote 1 shows BN > LN
# ─────────────────────────────────────────────────────────────────────
class PatchTSTEncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.BatchNorm1d(d_model)
        self.norm2 = nn.BatchNorm1d(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def _bn(self, norm: nn.BatchNorm1d, x: torch.Tensor) -> torch.Tensor:
        # BatchNorm1d expects (B, D, N); x is (B, N, D)
        return norm(x.transpose(1, 2)).transpose(1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B*M, N, D)
        attn_out, _ = self.attn(x, x, x)              # self-attention
        x = self._bn(self.norm1, x + attn_out)         # Add & Norm
        x = self._bn(self.norm2, x + self.ff(x))       # FFN + Add & Norm
        return x


# ─────────────────────────────────────────────────────────────────────
# Full PatchTST (supervised, channel-independent)
# ─────────────────────────────────────────────────────────────────────
class PatchTST(nn.Module):
    """
    Channel-independent PatchTST (supervised forecasting).

    Each of the M channels is processed independently through the same
    shared Transformer backbone (weight sharing across channels).

    Forward pass
    ------------
    1. Instance normalise each channel  (zero mean, unit std)
    2. Segment into N overlapping patches of length P
    3. Project patches → d_model  (learnable linear)
    4. Add learnable positional encoding
    5. Pass through n_layers Transformer encoder layers
    6. Flatten + linear head → forecast of length T
    7. Denormalise output
    """

    def __init__(
        self,
        M       : int,
        L       : int,
        T       : int,
        P       : int  = 16,
        S       : int  = 8,
        d_model : int  = 128,
        n_heads : int  = 16,
        n_layers: int  = 3,
        d_ff    : int  = 256,
        dropout : float = 0.2,
    ):
        super().__init__()
        self.M  = M
        self.L  = L
        self.T  = T
        self.P  = P
        self.S  = S
        self.N  = math.floor((L - P) / S) + 2   # number of patches

        # Patch projection: each patch of length P → d_model
        self.patch_proj = nn.Linear(P, d_model)

        # Learnable additive positional encoding (shape: 1 × N × d_model)
        self.pos_enc = nn.Parameter(torch.zeros(1, self.N, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)

        # Transformer encoder (n_layers stacked)
        self.encoder = nn.ModuleList([
            PatchTSTEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # Forecasting head: flatten N*d_model → T  (one per channel)
        self.head = nn.Linear(self.N * d_model, T)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x   : (B, M, L)
        out : (B, M, T)
        """
        B, M, _ = x.shape

        # ── 1. Instance normalisation ─────────────────────────────────
        mean = x.mean(dim=-1, keepdim=True)           # (B, M, 1)
        std  = x.std (dim=-1, keepdim=True) + 1e-5    # (B, M, 1)
        x_n  = (x - mean) / std

        # ── 2. Patch ──────────────────────────────────────────────────
        patches = make_patches(x_n, self.P, self.S)   # (B*M, N, P)

        # ── 3+4. Project + positional encoding ───────────────────────
        z = self.patch_proj(patches) + self.pos_enc   # (B*M, N, d_model)

        # ── 5. Transformer encoder ────────────────────────────────────
        for layer in self.encoder:
            z = layer(z)                              # (B*M, N, d_model)

        # ── 6. Flatten + forecast head ────────────────────────────────
        z_flat = z.reshape(B * M, -1)                 # (B*M, N*d_model)
        out    = self.head(z_flat)                    # (B*M, T)
        out    = out.reshape(B, M, self.T)            # (B, M, T)

        # ── 7. Denormalise ────────────────────────────────────────────
        out = out * std + mean
        return out


# ── Quick parameter count ─────────────────────────────────────────────
_tmp = PatchTST(M=M, L=L, T=96, P=P, S=S,
                d_model=D_MODEL, n_heads=N_HEADS,
                n_layers=N_LAYERS, d_ff=D_FF)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'PatchTST/42  N={_tmp.N} patches  params={n_params:,}')
del _tmp

PatchTST/42  N=42 patches  params=921,184


## Cell 7 — Training & evaluation helpers

In [ ]:
def run_epoch(model: nn.Module,
              loader: DataLoader,
              optimizer=None,
              is_train: bool = True) -> float:
    """
    One full pass through *loader*.
    Returns average MSE loss over all samples.
    Pass optimizer=None and is_train=False for validation / test.
    """
    model.train() if is_train else model.eval()
    total_loss = 0.0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)   # (B, M, L), (B, M, T)
            pred = model(x)                       # (B, M, T)
            loss = F.mse_loss(pred, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    """
    Compute MSE and MAE on a DataLoader.
    Returns (mse, mae) as Python floats.
    """
    model.eval()
    all_pred, all_true = [], []

    for x, y in loader:
        pred = model(x.to(DEVICE)).cpu()
        all_pred.append(pred)
        all_true.append(y)

    preds = torch.cat(all_pred)    # (N_samples, M, T)
    trues = torch.cat(all_true)

    mse = F.mse_loss(preds, trues).item()
    mae = (preds - trues).abs().mean().item()
    return mse, mae


print('Helpers defined.')

Helpers defined.


## Cell 8 — Checkpoint utilities

In [ ]:
def save_checkpoint(model: nn.Module,
                    optimizer,
                    epoch: int,
                    val_mse: float,
                    T: int,
                    tag: str = 'best') -> Path:
    """
    Save model state, optimizer state, epoch, and val_mse to disk.
    File name: patchtst_T{T}_{tag}.pt
    """
    path = CKPT_DIR / f'patchtst_T{T}_{tag}.pt'
    torch.save({
        'epoch'    : epoch,
        'val_mse'  : val_mse,
        'model'    : model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }, path)
    return path


def load_checkpoint(path: Path, model: nn.Module, optimizer=None):
    """
    Load a checkpoint saved by *save_checkpoint*.
    Returns the checkpoint dict.
    """
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt['optimizer'])
    print(f'Loaded checkpoint from {path}  '
          f'(epoch={ckpt["epoch"]}, val_mse={ckpt["val_mse"]:.4f})')
    return ckpt


def save_results(results: dict, fname: str = 'results_supervised.json'):
    """
    Persist the results dict as JSON in RESULTS_DIR.
    results format: {T: {'mse': float, 'mae': float}}
    """
    path = RESULTS_DIR / fname
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Results saved → {path}')


print('Checkpoint utilities defined.')
print(f'Checkpoints → {CKPT_DIR}')
print(f'Results     → {RESULTS_DIR}')

Checkpoint utilities defined.
Checkpoints → /content/drive/MyDrive/DL_Final_Project/checkpoints
Results     → /content/drive/MyDrive/DL_Final_Project/results


## Cell 9 — Train one horizon (core training loop)

In [ ]:
def train_one_horizon(data: np.ndarray, M: int, T: int) -> dict:
    """
    Full supervised training pipeline for a single prediction horizon T.

    Steps
    -----
    1. Build train / val / test loaders
    2. Instantiate PatchTST/42
    3. Train with Adam + ReduceLROnPlateau + early stopping
    4. Save best checkpoint whenever val MSE improves
    5. Reload best weights and evaluate on test set
    6. Return {mse, mae, best_epoch, history}

    Parameters
    ----------
    data : (N_timesteps, M)  raw unscaled array
    M    : number of channels
    T    : forecast horizon
    """
    print(f'\n{"="*64}')
    print(f'  Supervised PatchTST/42   T = {T}')
    print(f'  L={L}  P={P}  S={S}  '
          f'N={math.floor((L-P)/S)+2} patches')
    print(f'{"="*64}')

    # ── 1. Data ───────────────────────────────────────────────────────
    train_dl, val_dl, test_dl, _ = build_loaders(data, L, T)
    print(f'  Samples  train={len(train_dl.dataset):,}  '
          f'val={len(val_dl.dataset):,}  '
          f'test={len(test_dl.dataset):,}')

    # ── 2. Model ──────────────────────────────────────────────────────
    model = PatchTST(
        M=M, L=L, T=T, P=P, S=S,
        d_model=D_MODEL, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {n_params:,}')

    # ── 3. Optimiser & scheduler ──────────────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        patience=LR_PATIENCE,
        factor=0.5,
        min_lr=LR_MIN,
    )

    # ── 4. Training loop ──────────────────────────────────────────────
    best_val_mse   = float('inf')
    patience_count = 0
    best_epoch     = 0
    history        = []          # list of dicts, one per epoch

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()

        # Forward + backward pass over training set
        train_loss = run_epoch(model, train_dl, optimizer, is_train=True)

        # Validation (no gradients)
        val_mse, val_mae = evaluate(model, val_dl)

        # LR scheduler steps on val MSE
        scheduler.step(val_mse)
        current_lr = optimizer.param_groups[0]['lr']

        elapsed = time.time() - t0
        history.append({
            'epoch'     : epoch,
            'train_loss': train_loss,
            'val_mse'   : val_mse,
            'val_mae'   : val_mae,
            'lr'        : current_lr,
        })

        # ── Checkpoint: save whenever val MSE improves ────────────────
        if val_mse < best_val_mse:
            best_val_mse   = val_mse
            best_epoch     = epoch
            patience_count = 0
            ckpt_path = save_checkpoint(model, optimizer, epoch, val_mse, T)
        else:
            patience_count += 1

        # ── Logging (every 5 epochs and at epoch 1) ───────────────────
        if epoch == 1 or epoch % 5 == 0:
            print(
                f'  ep {epoch:3d}/{MAX_EPOCHS} | '
                f'train={train_loss:.4f} | '
                f'val MSE={val_mse:.4f} MAE={val_mae:.4f} | '
                f'lr={current_lr:.1e} | '
                f'{elapsed:.1f}s | '
                f'patience={patience_count}/{PATIENCE}'
            )

        # ── Early stopping ────────────────────────────────────────────
        if patience_count >= PATIENCE:
            print(f'\n  Early stop triggered at epoch {epoch}.'
                  f'  Best epoch = {best_epoch}  '
                  f'Best val MSE = {best_val_mse:.4f}')
            break

    # ── 5. Reload best weights and test evaluation ────────────────────
    print(f'\n  Reloading best checkpoint (epoch {best_epoch}) …')
    load_checkpoint(CKPT_DIR / f'patchtst_T{T}_best.pt', model)
    test_mse, test_mae = evaluate(model, test_dl)

    print(f'\n  ✓ TEST RESULT   MSE = {test_mse:.4f}   MAE = {test_mae:.4f}')

    return {
        'T'          : T,
        'mse'        : round(test_mse, 4),
        'mae'        : round(test_mae, 4),
        'best_epoch' : best_epoch,
        'best_val_mse': round(best_val_mse, 4),
        'history'    : history,
    }


print('Training function defined.')

Training function defined.


## Cell 10 — Run all horizons

In [ ]:
# ── Sweep over all four prediction horizons ───────────────────────────
# Each horizon trains an independent model, saves its best checkpoint,
# and appends its test results to *all_results*.

all_results = {}   # T → {mse, mae, best_epoch, history}

for T in HORIZONS:
    result = train_one_horizon(weather_data, M, T)
    all_results[str(T)] = {
        'mse'        : result['mse'],
        'mae'        : result['mae'],
        'best_epoch' : result['best_epoch'],
        'best_val_mse': result['best_val_mse'],
    }

# Persist results to disk
save_results(all_results, fname='results_supervised.json')

print('\nAll horizons complete.')


  Supervised PatchTST/42   T = 96
  L=336  P=16  S=8  N=42 patches
  Samples  train=36,361  val=4,825  test=10,081
  Parameters: 921,184
  ep   1/100 | train=0.3982 | val MSE=0.2998 MAE=0.3181 | lr=1.0e-04 | 101.2s | patience=0/10
  ep   5/100 | train=0.3478 | val MSE=0.2872 MAE=0.3083 | lr=1.0e-04 | 108.1s | patience=0/10
  ep  10/100 | train=0.3334 | val MSE=0.2846 MAE=0.3061 | lr=1.0e-04 | 107.8s | patience=3/10
  ep  15/100 | train=0.3227 | val MSE=0.2890 MAE=0.3095 | lr=5.0e-05 | 108.7s | patience=8/10

  Early stop triggered at epoch 17.  Best epoch = 7  Best val MSE = 0.2814

  Reloading best checkpoint (epoch 7) …
Loaded checkpoint from /content/drive/MyDrive/DL_Final_Project/checkpoints/patchtst_T96_best.pt  (epoch=7, val_mse=0.2814)

  ✓ TEST RESULT   MSE = 0.1900   MAE = 0.2467

  Supervised PatchTST/42   T = 192
  L=336  P=16  S=8  N=42 patches
  Samples  train=36,265  val=4,729  test=9,985
  Parameters: 1,437,376
  ep   1/100 | train=0.4510 | val MSE=0.3645 MAE=0.3651 | l

## Cell 11 — Results summary & comparison with paper

In [ ]:
# ── Paper targets for Supervised PatchTST/42, Weather dataset ─────────
paper = {
    96 : (0.152, 0.199),
    192: (0.197, 0.243),
    336: (0.249, 0.283),
    720: (0.320, 0.335),
}

print('\n' + '='*66)
print('  Supervised PatchTST/42 — Weather  (Table 4 replication)')
print('='*66)
print(f"{'T':>5}  "
      f"{'Paper MSE':>10}  {'Ours MSE':>10}  "
      f"{'Paper MAE':>10}  {'Ours MAE':>10}  "
      f"{'Best ep':>7}")
print('-'*66)

for T in HORIZONS:
    r        = all_results[str(T)]
    p_mse, p_mae = paper[T]
    mse_gap  = r['mse'] - p_mse
    mae_gap  = r['mae'] - p_mae
    mse_sym  = '▲' if mse_gap > 0 else '▼'
    mae_sym  = '▲' if mae_gap > 0 else '▼'
    print(
        f"  {T:5d}  "
        f"  {p_mse:>8.3f}    {r['mse']:>8.3f} {mse_sym}{abs(mse_gap):.3f}  "
        f"  {p_mae:>8.3f}    {r['mae']:>8.3f} {mae_sym}{abs(mae_gap):.3f}  "
        f"  {r['best_epoch']:>5d}"
    )

print('='*66)
print('▲ = above paper target  ▼ = below (better than paper)')

# Also print saved file locations
print(f'\nCheckpoints : {CKPT_DIR}')
print(f'Results JSON: {RESULTS_DIR / "results_supervised.json"}')